# 用于文档处理的语义分块

## 概述

该代码实现了一种语义分块方法，用于处理和检索 PDF 文档中的信息，[首先由 Greg Kamradt 提出](https://youtu.be/8OJC21T2SL4?t=1933)，随后[在 LangChain 中实现](https://python.langchain.com/docs/how_to/semantic-chunker/)。与基于固定字符或字数分割文本的传统方法不同，语义分块旨在创建更有意义和上下文感知的文本片段。

## 动机

传统的文本分割方法经常在任意点破坏文档，可能会破坏信息和上下文的流动。语义分块通过尝试在更自然的断点处分割文本来解决此问题，从而保持每个块内的语义一致性。

## 关键组件

1. PDF 处理和文本提取
2. 使用LangChain的SemanticChunker进行语义分块
3. 使用 FAISS 和 OpenAI 嵌入支持存储创建
4. 用于查询已处理文档的检索器设置

## 方法详细信息

### 文档预处理

1. 使用自定义 `read_pdf_to_string` 函数读取 PDF 并将其转换为字符串。

### 语义分块

1. 利用LangChain的`SemanticChunker`和OpenAI嵌入。
2. 提供三种断点类型：
   -“百分位数”：以大于 X 百分位数的差异进行分割。
   - 'standard_deviation'：以大于 X 标准差的差异进行分割。
   - '四分位距'：使用四分位距来确定分割点。
3. 在此实现中，使用“百分位数”方法，阈值为 90。

### 支持存储创建

1. OpenAI 嵌入用于创建语义块的向量表示。
2. 根据这些嵌入创建 FAISS 存储存储，以实现高效的相似性搜索。

### 搜索器设置

1. 检索器配置为获取给定查询的前 2 个最相关的块。

## 主要特点

1. 上下文感知拆分：尝试保持块内的语义连贯性。
2. 灵活配置：允许不同的断点类型和阈值。
3. 与高级 NLP 工具集成：使用 OpenAI 嵌入进行分块和检索。

## 这种方法的好处

1. 提高连贯性：块更有可能包含完整的想法或想法。
2. 更好的检索相关性：通过保留上下文，可以提高检索准确性。
3. 适应性：分块方法可以根据文档的性质和检索的需要进行调整。
4. 更好理解的潜力：法学硕士或下游任务可能会通过更连贯的文本片段表现得更好。

## 实施细节

1. 使用 OpenAI 的嵌入进行语义分块过程和最终向量表示。
2. 使用 FAISS 创建高效的块可搜索索引。
3. 搜索器设置为返回前 2 个最相关的块，可以根据需要进行调整。

## 用法示例

该代码包含一个测试查询：“气候变化的主要原因是什么？”。这演示了如何使用语义分块和检索系统从处理的文档中查找相关信息。

## 结论

语义分块代表了检索系统文档处理的高级方法。通过尝试保持文本片段内的语义一致性，它有可能提高检索信息的质量并增强下游 NLP 任务的性能。此技术对于处理维护上下文至关重要的长而复杂的文档（例如科学论文、法律文档或综合报告）特别有价值。

<div style="text-align: center;">

<img src="../images/semantic_chunking_comparison.svg" alt="Self RAG" style="width:100%; height:auto;">
</div>

# 包安装和导入

下面的单元格安装运行此笔记本所需的所有必需软件包。

In [ ]:
# Install required packages
!pip install langchain-experimental langchain-openai python-dotenv

In [ ]:
# Clone the repository to access helper functions and evaluation modules
!git clone https://github.com/NirDiamant/RAG_TECHNIQUES.git
import sys
sys.path.append('RAG_TECHNIQUES')
# If you need to run with the latest data
# !cp -r RAG_TECHNIQUES/data .

In [57]:
import os
import sys
from dotenv import load_dotenv

# Original path append replaced for Colab compatibility
from helper_functions import *
from evaluation.evalute_rag import *

from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings

# Load environment variables from a .env file
load_dotenv()

# Set the OpenAI API key environment variable
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')



### 定义文件路径

In [ ]:
# Download required data files
import os
os.makedirs('data', exist_ok=True)

# Download the PDF document used in this notebook
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf


In [3]:
path = "data/Understanding_Climate_Change.pdf"

### 读取 PDF 到字符串

In [18]:
content = read_pdf_to_string(path)

### 断点类型： 
* 'percentile'：计算句子之间的所有差异，然后将任何大于 X 百分位数的差异进行分割。
* 'standard_deviation'：任何大于 X 个标准差的差异都会被分割。
* 'interquartile'：四分位距离用于分割块。

In [51]:
text_splitter = SemanticChunker(OpenAIEmbeddings(), breakpoint_threshold_type='percentile', breakpoint_threshold_amount=90) # chose which embeddings and breakpoint type and threshold to use

### 将原始文本拆分为语义块

In [53]:
docs = text_splitter.create_documents([content])

### 创建支持存储和搜索器

In [54]:
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embeddings)
chunks_query_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

### 测试搜索器

In [ ]:
test_query = "What is the main cause of climate change?"
context = retrieve_context_per_question(test_query, chunks_query_retriever)
show_context(context)

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--semantic-chunking)